# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('C:\\Users\\SAM\\OneDrive\\Documents\\DV\\data-viz-class-material\\data\\netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [4]:
import plotly.express as px
import pandas as pd

# Assume 'df' is your loaded Netflix catalogue DataFrame
# df = pd.read_csv('data/netflix_catalogue.csv')

# 1. Create a 'decade' column using the exact formula provided in Screenshot 2026-05-22 121037.png
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# 2. Filter the data to include only the specified most common ratings
target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(target_ratings)]

# 3. Aggregate data by rating and decade to get the counts for the heatmap matrix
heatmap_data = pd.crosstab(
    index=df_filtered['rating'], 
    columns=df_filtered['decade']
).reindex(index=target_ratings)

# 4. Build the heatmap using Plotly Express
fig = px.imshow(
    heatmap_data,
    labels=dict(x="Release Decade", y="Content Rating", color="Number of Titles"),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale='Blues',   
    text_auto=True,                   
    title='<b>Netflix Catalogue: TV-MA Dominates the 2010s and 2020s Streaming Boom</b><br>'
          '<sup>Analysis of title counts across historical release decades and primary content ratings.</sup>'
)

# 5. Fine-tune layout adjustments (Fixed Colorbar Property)
fig.update_layout(
    xaxis_title='Release Decade',
    yaxis_title='Content Rating',
    coloraxis_colorbar=dict(title='Titles')
)

# Display the chart
fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [11]:
import plotly.graph_objects as go
import pandas as pd

# 1. Load the raw workspace file using your absolute path
df_raw = pd.read_csv('C:\\Users\\SAM\\OneDrive\\Documents\\DV\\data-viz-class-material\\data\\netflix_catalogue.csv')

# 2. Clean data and filter to Movies strictly between 2015 and 2022
# (No 'date_added' here anymore, preventing the KeyError!)
df_clean = df_raw.dropna(subset=['added_year']).copy()
df_clean['added_year'] = df_clean['added_year'].astype(int)
df_movies = df_clean[(df_clean['type'] == 'Movie') & (df_clean['added_year'].between(2015, 2022))]

# 3. Group data and count additions per year
yearly_counts = df_movies.groupby('added_year').size().reset_index(name='count')

# 4. Convert lists to string types to prevent the squished layout timeline bug
years_seq = [str(int(yr)) for yr in yearly_counts['added_year']]
counts_seq = [int(c) for c in yearly_counts['count']]

# Identify metrics for peak annotation
max_idx = yearly_counts['count'].idxmax()
max_year_str = years_seq[max_idx]
max_count = counts_seq[max_idx]

# 5. Append the Native Total Bar configuration
years_seq.append('Total')
counts_seq.append(0)
measure_types = ['relative'] * len(yearly_counts) + ['total']

# 6. Build Waterfall Chart using a fresh figure variable name
fig_waterfall_final = go.Figure(go.Waterfall(
    name="Movie Additions",
    orientation="v",
    measure=measure_types,
    x=years_seq,      
    y=counts_seq,     
    textposition="outside",
    text=[f"+{c}" if m == 'relative' else "Total" for c, m in zip(counts_seq, measure_types)],
    decreasing=dict(marker=dict(color="Red")),  
    increasing=dict(marker=dict(color="MediumSeaGreen")), 
    totals=dict(marker=dict(color="RoyalBlue"))
))

# 7. Title requirements & axis configuration
fig_waterfall_final.update_layout(
    title=f'<b>Netflix Movie Library: Peak Expansion in {max_year_str} Drives Massive Catalogue Growth</b><br>'
          f'<sup>Year-over-year movie additions from 2015 to 2022 with final cumulative market library size.</sup>',
    yaxis_title="Number of Movies",
    waterfallgap=0.3
)

# Hard-override x-axis category settings to prevent the data points from stacking on top of each other
fig_waterfall_final.update_xaxes(type='category', title_text="Year Added")

# 8. Add peak single addition label annotation
fig_waterfall_final.add_annotation(
    x=max_year_str,
    y=max_count,
    text=f"<b>Peak Growth Year</b><br>+{max_count} Movies Added",
    showarrow=True,
    arrowhead=2,
    arrowcolor="black",
    ax=0,
    ay=-40,
    bordercolor="black",
    borderwidth=1,
    borderpad=4,
    bgcolor="white",
    opacity=0.9
)

# Render output cleanly
fig_waterfall_final.show()